### Making the 735 kv network converge

In [2]:
%cd ~/Documents/McGill/capstone/quebec-pypsa-earth

/home/etienne/Documents/McGill/capstone/quebec-pypsa-earth


In [3]:
!python network/run_pf.py --network network/networks_current/elec_solved.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

!mv network/networks_current/elec_solved_pf.nc etienne_tests/results/elec_315kv_pf.nc

Loading solved network: network/networks_current/elec_solved.nc
INFO:pypsa.io:Imported network elec_solved.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers
  205 buses, 168 snapshots
  Copied LOPF dispatch (generators.p) into p_set
  Copied LOPF dispatch (storage_units.p) into p_set
  Copied LOPF dispatch (links.p0) into p_set

Sub-networks:
attribute  carrier slack_bus           obj
SubNetwork                                
0               AC       339  SubNetwork 0
1               DC        65  SubNetwork 1
2               DC       161  SubNetwork 2
3               DC      1015  SubNetwork 3
4               DC      2206  SubNetwork 4
5               DC      3549  SubNetwork 5
  SubNetwork 0 (AC, 200 buses): slack_bus=339, slack generator(s)=['339 slack-placeholder']
  SubNetwork 1 (DC, 1 buses): slack_bus=65, slack generator(s)=[]
  SubNetwork 2 (DC, 1 buses): slack_bus=161, slack generator(s)=['161 load_shedding']
  SubNetwork 3 (DC, 1 buses): sl

What happens when we run AC power flow on our 315kv network?  

Input: network/networks_current/elec_solved.nc  
Output: looking at v_mag_pu, and converging snapshots (3/168) clearly the model does not converge. We also observe load shedding.  

What happens if we apply the fixes that we applied to the 735kV network to this one?



In [4]:
!python network/correct_sending_end_circuits.py --network network/networks_current/elec_solved.nc --output etienne_tests/results/elec_315kv.nc
!python network/run_pf.py --network etienne_tests/results/elec_315kv.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

Loading network: network/networks_current/elec_solved.nc
INFO:pypsa.io:Imported network elec_solved.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers
Corrected circuit count on 7 sending-end lines:
      num_parallel_old  r_old   x_old  b_old  s_nom_old  num_parallel_new  r_new   x_new  b_new  s_nom_new
Line                                                                                                      
2408               1.0   5.01  165.21    0.0    5242.78               3.0   1.67   55.07   0.01   15728.35
162                1.0   4.57  150.67    0.0    5682.19               3.0   1.52   50.22   0.01   17046.58
3756               1.0   6.23  205.45    0.0    4380.16               3.0   2.08   68.48   0.01   13140.49
1049               1.0   5.63  185.91    0.0    4746.22               1.5   3.76  123.94   0.00    7119.33
1448               1.0   5.68  187.35    0.0    4716.22               1.5   3.78  124.90   0.00    7074.33
156               

Okay, this is better. It still doesn't converge well, but we have convergence for 10/168 snapshots, which is better than 3. Let's continue.

In [5]:
!python network/apply_series_compensation.py --network etienne_tests/results/elec_315kv.nc --in-place
!python network/run_pf.py --network etienne_tests/results/elec_315kv.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

Loading network: etienne_tests/results/elec_315kv.nc
INFO:pypsa.io:Imported network elec_315kv.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers
Applied 70% series compensation to 19 lines:
  2408 (114-1291 (500.6km, sole supply path to bus 1291)): x 55.07 -> 16.52 ohm
  880 (312-308 circuit 1 (433.4km)): x 143.02 -> 42.91 ohm
  1935 (312-308 circuit 2 (409.5km)): x 135.14 -> 40.54 ohm
  2410 (312-308 circuit 3 (409.6km)): x 135.15 -> 40.55 ohm
  1536 (308-2944 (256.2km)): x 84.54 -> 25.36 ohm
  1538 (308-1081 (266.1km)): x 87.80 -> 26.34 ohm
  1936 (308-136 (348.0km)): x 114.85 -> 34.45 ohm
  163 (457-312 circuit 1 (344.7km)): x 113.74 -> 34.12 ohm
  1655 (457-312 circuit 2 (344.6km)): x 113.73 -> 34.12 ohm
  2386 (457-312 circuit 3 (378.2km)): x 124.80 -> 37.44 ohm
  162 (340-457 (456.6km, highest transmission angle on the network)): x 50.22 -> 15.07 ohm
  156 (66-457 circuit 1 (432.7km, high transmission angle)): x 95.19 -> 28.56 ohm
  1938 (66-457

Applying series compensation gives convergence of 106/168 snapshots. The voltages look off though, they are way too high. This is normal, since not all snapshots converge, so some voltages can be extreme.

In [13]:
import pypsa, numpy as np, pandas as pd
n = pypsa.Network("etienne_tests/results/elec_315kv_shunt_pf.nc")
v = n.buses_t.v_mag_pu
ok = ((v > 0.5) & (v < 1.5)).all(axis=1)          # drops the diverged snapshots
vo = v[ok]
print(f"usable snapshots: {ok.sum()} / {len(ok)}")
print(f"voltage range: {vo.min().min():.3f} - {vo.max().max():.3f} pu")
print(f"buses ever below 0.95: {(vo.min() < 0.95).sum()} / {vo.shape[1]}")
print(f"buses ever above 1.05: {(vo.max() > 1.05).sum()} / {vo.shape[1]}")

# worst offenders
print("\nlowest buses:\n", vo.min().sort_values().head(8).round(3))
print("\nhighest buses:\n", vo.max().sort_values(ascending=False).head(8).round(3))

# when do violations occur?
low = (vo < 0.95).sum(axis=1)
high = (vo > 1.05).sum(axis=1)
print("\nhours with most low-voltage buses:\n", low.sort_values(ascending=False).head(5))
print("\nhours with most high-voltage buses:\n", high.sort_values(ascending=False).head(5))

INFO:pypsa.io:Imported network elec_315kv_shunt_pf.nc has buses, carriers, generators, lines, links, loads, shunt_impedances, storage_units, transformers


usable snapshots: 126 / 168
voltage range: 0.861 - 1.494 pu
buses ever below 0.95: 15 / 205
buses ever above 1.05: 117 / 205

lowest buses:
 Bus
800     0.861
263     0.861
271     0.896
45      0.910
1664    0.916
194     0.920
1715    0.923
2892    0.926
dtype: float64

highest buses:
 Bus
1810    1.494
1806    1.471
1804    1.391
1803    1.369
1800    1.342
311     1.322
1638    1.312
312     1.312
dtype: float64

hours with most low-voltage buses:
 snapshot
2022-01-03 17:00:00    15
2022-01-03 18:00:00    14
2022-01-04 09:00:00    14
2022-01-04 08:00:00    14
2022-01-04 10:00:00    11
dtype: int64

hours with most high-voltage buses:
 snapshot
2022-01-01 19:00:00    117
2022-01-06 02:00:00    117
2022-01-05 23:00:00    117
2022-01-06 03:00:00    117
2022-01-01 08:00:00    117
dtype: int64


In [11]:
!python network/add_shunt_impedance.py --network etienne_tests/results/elec_315kv_pf.nc --output etienne_tests/results/elec_315kv_shunt.nc

INFO:pypsa.io:Imported network elec_315kv_pf.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers

--- Preparing for AC PF (no-op for DC/lpf) ---
1. Transformer r = x / 30 (generic X/R, not measured) on 22 transformers
2. Load q_set = p_set * tan(acos(0.95)) on 647 loads (generic 0.95 lagging power factor assumption)
1b. Added 0 zero-dispatch placeholder generator(s) on storage-only buses (0 MW combined, sized to match), so they're eligible for PV classification below
2b. Generator q_set = p_nom * tan(acos(0.9)) on 88 real generators (generic 0.9 PF reactive capability, fixed to installed capacity not dispatch)
3. Marked 45 / 53 generator buses PV (p_nom >= 0 MW and load < 100% of local generation; largest plain Generator there each -- including the storage-pv-placeholders added in step 1b, now eligible on the same terms as real generators). Remaining generator buses stay PQ (too small and/or too load-dominated to plausibly regulate voltage in practice).

In [12]:
!python network/run_pf.py --network etienne_tests/results/elec_315kv_shunt.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

Loading solved network: etienne_tests/results/elec_315kv_shunt.nc
INFO:pypsa.io:Imported network elec_315kv_shunt.nc has buses, carriers, generators, lines, links, loads, shunt_impedances, storage_units, transformers
  205 buses, 168 snapshots

Sub-networks:
attribute  carrier slack_bus           obj
SubNetwork                                
0               AC       339  SubNetwork 0
1               DC        65  SubNetwork 1
2               DC       161  SubNetwork 2
3               DC      1015  SubNetwork 3
4               DC      2206  SubNetwork 4
5               DC      3549  SubNetwork 5
  SubNetwork 0 (AC, 200 buses): slack_bus=339, slack generator(s)=['339 slack-placeholder']
  SubNetwork 1 (DC, 1 buses): slack_bus=65, slack generator(s)=['65 pf-slack-placeholder']
  SubNetwork 2 (DC, 1 buses): slack_bus=161, slack generator(s)=['161 load_shedding']
  SubNetwork 3 (DC, 1 buses): slack_bus=1015, slack generator(s)=['1015 load_shedding']
  SubNetwork 4 (DC, 1 buses): slack_bus=

In [ ]:
cand = grid[grid.converged == full]
    if cand.empty:
        cand = grid[grid.converged == grid.converged.max()]
        print(f"[warn] no scale pair converges on all {full} snapshots; "
              f"using the best available ({int(grid.converged.max())}/{full})")
    best = cand.sort_values(["out_buses", "out_bus_hours"]).iloc[0]

Added to add_shunt_impedance main().

In [14]:
import pypsa
n = pypsa.Network("etienne_tests/results/elec_315kv_shunt_pf.nc")
buses = ["1800", "1803", "1804", "1806", "1810", "311"]
print(n.generators[n.generators.bus.isin(buses)][["bus", "carrier", "p_nom", "control", "q_set"]])
print(n.loads[n.loads.bus.isin(buses)][["bus", "p_set"]])
print(n.shunt_impedances[n.shunt_impedances.bus.isin(buses)])
L = n.lines[n.lines.bus0.isin(buses) | n.lines.bus1.isin(buses)]
print(L[["bus0", "bus1", "v_nom", "length", "num_parallel", "b"]])

INFO:pypsa.io:Imported network elec_315kv_shunt_pf.nc has buses, carriers, generators, lines, links, loads, shunt_impedances, storage_units, transformers


                              bus        carrier       p_nom control  \
Generator                                                              
1788 ror-Premiere-Chute      1810            ror  124.000000      PQ   
1783 ror-Rapides-des-Iles    1810            ror  176.000000      PQ   
2186 ror-Rapides-des-Quinze  1810            ror  111.000000      PQ   
311 load_shedding             311  load_shedding    0.462519      PQ   
1810 load_shedding           1810  load_shedding  696.891228      PQ   
1800 load_shedding           1800  load_shedding    0.365493      PQ   
1803 load_shedding           1803  load_shedding    0.336928      PQ   
1804 load_shedding           1804  load_shedding    0.513315      PQ   
1806 load_shedding           1806  load_shedding   95.392172      PQ   

                                 q_set  
Generator                               
1788 ror-Premiere-Chute      60.055941  
1783 ror-Rapides-des-Iles    85.240690  
2186 ror-Rapides-des-Quinze  53.759754  
31


| Test | Converged snapshots |
| :--- | :---: |
| Base | 3/168 |
| Correct_sending_end | 10/168 |
| Compensation | 106/168 |
| Compensate 3535 | 110/168 |
| Compensate 161 | 154/168 |
| Compensate 1653 | 156/168 |
| Compensate 2387 | 157/168 |
| Compensate 159 | 157/168 | (not applied)
| Compensate 2398 | 157/168 | (not applied)
| Add shunts | 161/168 | (reduced the SCALE_GRID table)
